# WM-811K 1단계: 대용량 pickle 전처리

Kaggle의 `qingyi/wm811k-wafer-map`에서 `LSWMD.pkl`을 Colab으로 직접 내려받고, 라벨이 있는 웨이퍼맵을 64×64 `uint8` 배열로 변환합니다. 2GB 원본을 직접 업로드할 필요가 없습니다. 이 단계는 GPU보다 RAM이 중요하므로 **고용량 RAM** 런타임이 보이면 선택하세요.

실행 전 Colab 왼쪽의 열쇠(Secrets) 메뉴에 `KAGGLE_API_TOKEN`을 추가하고 이 노트북의 접근을 허용하세요. 토큰은 Kaggle `Settings → API → Generate New Token`에서 발급합니다.

In [ ]:
!pip install -q -U kagglehub

from google.colab import drive, files, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys
import kagglehub

token = userdata.get('KAGGLE_API_TOKEN')
assert token, 'Colab Secrets에 KAGGLE_API_TOKEN을 추가하고 노트북 접근을 허용하세요.'
os.environ['KAGGLE_API_TOKEN'] = token

download_root = Path('/content/wm811k_kaggle')
download_root.mkdir(parents=True, exist_ok=True)
downloaded = Path(kagglehub.dataset_download(
    'qingyi/wm811k-wafer-map',
    path='LSWMD.pkl',
    output_dir=str(download_root),
))
candidates = ([downloaded] if downloaded.is_file() else []) + list(download_root.rglob('LSWMD.pkl'))
assert candidates, f'다운로드 후 LSWMD.pkl을 찾을 수 없습니다: {downloaded}'
PICKLE_PATH = candidates[0]

drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
meminfo = Path('/proc/meminfo').read_text(encoding='ascii').splitlines()
available_kb = next(int(line.split()[1]) for line in meminfo if line.startswith('MemAvailable:'))
available_gb = available_kb / 1024**2
print(f'사용 가능한 RAM: {available_gb:.2f} GB')
assert available_gb >= 8.0, '가용 RAM이 8GB 미만입니다. 고용량 RAM 런타임으로 다시 연결하세요.'
print('원본:', PICKLE_PATH, f'{PICKLE_PATH.stat().st_size/1024**3:.3f} GB')
print('출력:', OUTPUT_DIR)

## 전처리 스크립트 업로드
프로젝트의 `코드/wm811k/preprocess_wm811k.py` 파일 하나를 선택하세요.

In [ ]:
uploaded = files.upload()
assert 'preprocess_wm811k.py' in uploaded, 'preprocess_wm811k.py를 선택하세요.'
SCRIPT_PATH = Path('/content/preprocess_wm811k.py')
SCRIPT_PATH.write_bytes(uploaded['preprocess_wm811k.py'])
subprocess.run([sys.executable, str(SCRIPT_PATH), '--input', str(PICKLE_PATH), '--output-dir', str(OUTPUT_DIR), '--image-size', '64'], check=True)
print('전처리 완료')

## 결과 확인 및 작은 영수증 ZIP 다운로드
대형 `wafer_maps_64.npy`는 Drive에 유지합니다. 요약 JSON·CSV만 ZIP으로 내려받아 Codex에 전달하세요.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'dataset_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert (OUTPUT_DIR / 'wafer_maps_64.npy').is_file()
assert (OUTPUT_DIR / 'labels.npy').is_file()
receipt_dir = Path('/content/wm811k_receipt')
receipt_dir.mkdir(exist_ok=True)
for name in ('dataset_summary.json', 'label_mapping.json', 'class_distribution.csv', 'labeled_metadata.csv', 'invalid_rows.csv'):
    source = OUTPUT_DIR / name
    if source.exists():
        shutil.copy2(source, receipt_dir / name)
archive = shutil.make_archive('/content/wm811k_preprocessing_receipt', 'zip', receipt_dir)
print('Drive 학습 데이터:', OUTPUT_DIR)
print('전달용 요약:', archive)
files.download(archive)